In [1]:
# Step 1: Install & Import
!pip install spacy

import spacy
from spacy.matcher import PhraseMatcher

# Step 2: Load spaCy model (better similarity use 'md' or 'lg')
nlp = spacy.load("en_core_web_md")

# Step 3: Define Categories & Keywords
categories = {
    "Billing Team": ["payment failed", "billing issue", "transaction error"],
    "Technical Team": ["unable to login", "account issue", "technical error", "login"],
    "Refund Team": ["refund", "return order", "money back"],
    "Sales Team": ["buy", "purchase", "order product", "discount"]
}

# Step 4: Create PhraseMatcher
matcher = PhraseMatcher(nlp.vocab)
for dept, phrases in categories.items():
    patterns = [nlp(text) for text in phrases]
    matcher.add(dept, patterns)

# Step 5: Tickets (covering all teams)
tickets = [
    "My payment failed",                          # Billing Team
    "Unable to login to my account",              # Technical Team
    "Need refund for my order",                   # Refund Team
    "Do you have any discount for mobiles?"       # Sales Team
]

# Step 6: Router Loop
for ticket in tickets:
    doc = nlp(ticket)
    print("\n===========================================================")
    print("CUSTOMER SUPPORT TICKET ROUTER")
    print("===========================================================\n")
    print("Customer Ticket:\n   ", ticket)

    # Named Entities
    print("\nNamed Entities:")
    if doc.ents:
        for ent in doc.ents:
            print(ent.text, ent.label_)
    else:
        print("No named entities detected.")

    # Matcher Results
    matches = matcher(doc)
    print("\nMatcher Results:")
    if matches:
        for match_id, start, end in matches:
            dept = nlp.vocab.strings[match_id]
            print(f"{dept.upper()}: {doc[start:end]}")
    else:
        print("No matcher keywords found.")

    # Semantic Similarity
    print("\nSemantic Similarity:")
    sims = {dept: max([nlp(ticket).similarity(nlp(p)) for p in phrases])
            for dept, phrases in categories.items()}
    for dept, score in sims.items():
        print(f"{dept:15} {score:.4f}")

    # Final Scores (matcher boost)
    final_scores = {}
    for dept, score in sims.items():
        boost = 0.05 if any(p in ticket.lower() for p in categories[dept]) else 0
        final_scores[dept] = score + boost

    print("\nFinal Scores:")
    for dept, score in final_scores.items():
        print(f"{dept:15} {score:.4f}")

    best_dept = max(final_scores, key=final_scores.get)
    confidence = final_scores[best_dept] * 100

    print("\n-----------------------------------------------------------")
    print(f"FINAL DEPARTMENT : {best_dept}")
    print(f"CONFIDENCE : {confidence:.2f}%")
    print("-----------------------------------------------------------")



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



CUSTOMER SUPPORT TICKET ROUTER

Customer Ticket:
    My payment failed

Named Entities:
No named entities detected.

Matcher Results:
BILLING TEAM: payment failed

Semantic Similarity:
Billing Team    0.9228
Technical Team  0.7224
Refund Team     0.7586
Sales Team      0.5169

Final Scores:
Billing Team    0.9728
Technical Team  0.7224
Refund Team     0.7586
Sales Team      0.5169

-----------------------------------------------------------
FINAL DEPARTMENT : Billing Team
CONFIDENCE : 97.28%
-----------------------------------------------------------

CUSTOMER SUPPORT TICKET ROUTER

Customer Ticket:
    Unable to login to my account

Named Entities:
No named entities detected.

Matcher Results:
TECHNICAL TEAM: login

Semantic Similarity:
Billing Team    0.6820
Technical Team  0.9040
Refund Team     0.7334
Sales Team      0.5082

Final Scores:
Billing Team    0.6820
Technical Team  0.9540
Refund Team     0.7334
Sales Team      0.5082

---------------------------------------------------